# Two QCQP tracks for rotations

`QcqpProblem` converts supported factor graphs into quadratic costs and constraints over direct vector or matrix values. Rotation problems have two deliberately different tracks: an exact homogeneous Rot2 lift at $D=1$, and a Stiefel/Burer--Monteiro formulation at $D\ge N$. This tutorial develops their mathematics, explains gauge and reflection ambiguities, and maps each claim to a unit test.

GTSAM Copyright 2010-2022, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/constrained/doc/QcqpProblem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [ ]:
import gtsam
import numpy as np

## 1. The exact $D=1$ Rot2 lift

For $R\in SO(2)$, the homogeneous variable is

$$x = [x_0,\operatorname{vec}(R)^\top]^\top\in\mathbb R^5,$$

where `vec` is column-major. Quadratic equalities encode $x_0^2=1$, $\det R=1$, and $RR^\top=I$. The first equality fixes only the magnitude of $x_0$: both $x$ and $-x$ satisfy all homogeneous quadratic constraints and produce the same homogeneous between-factor costs.

A hard Frobenius prior removes this sign ambiguity with the linear equality

$$x=[1,\operatorname{vec}(M)^\top]^\top.$$

Negating a feasible Rot2 lift changes this equality residual by $-2x$. Since $\|x\|^2=1+\|R\|_F^2=3$, its violation norm is exactly $2\sqrt3$. This track is exact, but currently supports only Rot2 and only hard priors.

## 2. The $D\ge N$ Stiefel track

For a rotation $R\in SO(N)$ and $D\ge N$, define the canonical lift

$$X=R^\top S,\qquad S=[I_N\;0]\in\mathbb R^{N\times D}.$$

The row-space quadratic constraints enforce

$$XX^\top=I_N,$$

so $X$ lies on the row Stiefel manifold $\mathrm{St}(N,D)$. There are $N$ unit-row constraints and $N(N-1)/2$ row-orthogonality constraints. They do **not** encode determinant $+1$ or force the padded columns to remain zero; consequently this is a relaxation of the canonical rotation embedding. Rot2 supports every $D\ge2$, and Rot3 supports every $D\ge3$.

## 3. Between costs and the right-$O(D)$ gauge

A matrix-form Frobenius between factor with measurement $M$ contributes

$$E_{ij}=\frac{1}{2\sigma^2}\|X_j-M^\top X_i\|_F^2.$$

For every $G\in O(D)$, replacing all variables in a connected component by $X_iG$ preserves both feasibility and every between cost:

$$\|(X_j-M^\top X_i)G\|_F=\|X_j-M^\top X_i\|_F.$$

Thus an unanchored component determines relative rotations but not an absolute lifted frame. This common right action is the reference-frame, or gauge, freedom of the matrix track.

## 4. Why matrix-form priors are deferred

The seemingly direct prior with fixed target $\bar X=[M^\top\;0]$,

$$\frac{1}{2\sigma^2}\|X_i-\bar X\|_F^2,$$

agrees with the original Frobenius error on canonical lifts, but it is not invariant under $X_i\mapsto X_iG$. Its informative term is linear in $X_i$ and therefore cannot be represented using only the Burer--Monteiro Gram matrix. This branch consequently rejects all matrix-form `FrobeniusPrior` conversions.

A future BM-compatible lowering can introduce an anchor block $X_a$ and represent the prior as the between cost

$$\frac{1}{2\sigma^2}\|X_i-M^\top X_a\|_F^2.$$

The common transformation $(X_i,X_a)\mapsto(X_iG,X_aG)$ leaves this cost unchanged, so it is quadratic in Gram-matrix blocks and compatible with a certified staircase. That anchor-block lowering and the corresponding gauge-aware recovery belong to the future BM solver branch.

## 5. Reflections, initialization, and extraction

When $D=N$, $XX^\top=I$ describes $O(N)$, which has rotation and reflection components. Initializing with `InsertQcqpValue` starts in the canonical rotation component; a manifold-constrained local method initialized in the reflected component may remain there. For $D>N$, the Stiefel manifold no longer has this particular two-component obstruction.

`ExtractQcqpValues<T,D>` accepts only exact $N\times D$ matrix slices and applies `ClosestTo` to the transpose of the leading $N\times N$ block. This is reliable for canonical values or after the caller has chosen a gauge. Otherwise it is deliberately best-effort: a common gauge changes the extracted absolute rotations, although relative rotations remain invariant.

## 6. C++ construction patterns

The exact track uses `columnDimension=1` and a constrained prior:

```cpp
NonlinearFactorGraph graph;
graph.emplace_shared<FrobeniusPrior<Rot2>>(
    x0, measured.matrix(), noiseModel::Constrained::All(4));
QcqpProblem exactProblem(graph, 1);
```

The current Stiefel track supports gauge-invariant between factors:

```cpp
NonlinearFactorGraph graph;
graph.emplace_shared<FrobeniusBetweenFactor<Rot2>>(x0, x1, relative);
QcqpProblem stiefelProblem(graph, 3);

Values initial;
InsertQcqpValue<Rot2, 3>(x0, initialR0, &initial);
InsertQcqpValue<Rot2, 3>(x1, initialR1, &initial);
```

The row-space `QpCost` constructor represents $\tfrac12\sum_{ij}\operatorname{tr}(X_i^\top Q_{ij}X_j)$. Adding a matrix-form `FrobeniusPrior` currently throws; the future anchor-block lowering will produce another row-space between cost instead of an affine fixed-target cost.

## API and support summary

| Type | Exact vector track | Matrix/Stiefel track | Priors |
|---|---|---|---|
| Rot2 | $D=1$ | every $D\ge2$ | hard at $D=1$; matrix prior deferred |
| Rot3 | not implemented | every $D\ge3$ | matrix prior deferred |

The principal entry points are `QcqpProblem(graph, columnDimension)`, `InsertQcqpValue<T,D>`, `InsertQcqpConstraints<T,D>`, and `ExtractQcqpValues<T,D>`. Unsupported nonlinear factors retain the default behavior of throwing during conversion.

## References

- [QcqpProblem API](../QcqpProblem.h)
- [QpCost API](../QpCost.h)
- [Frobenius factors](../../slam/FrobeniusFactor.h)
- [QCQP Python example](../../../python/gtsam/examples/QcqpProblemExample.ipynb)
- [QP Python example](../../../python/gtsam/examples/QpProblemExample.ipynb)